# MUSSA: train the two AI models on ARCADE (segments + narrowings)

Runs on Google Colab or Kaggle with a free GPU. It trains only on ARCADE, a public CC0 dataset (https://zenodo.org/records/10390295). No patient data is used or uploaded unless you deliberately switch on the optional fine-tuning cell (read its warning first).

**Colab:** Runtime > Change runtime type > GPU.
**Kaggle:** in the notebook settings turn on the GPU accelerator and Internet, and add `mussa-training-kit.zip` as a dataset (Add data > Upload).

Steps: install, download and verify ARCADE, load the kit, train the **segment model**, train the **stenosis model**, check both, download **four files** and copy them to `C:\Users\<you>\MUSSA-data\models\`:
`segments.onnx`, `segments.card.json`, `stenosis.onnx`, `stenosis.card.json`.

Both trainings stop early when the validation score stops improving, so they may finish before the epoch limit. Add `--smoke` to a command first for a one-minute pipeline check.

Educational research model. Not a certified medical device.

In [ ]:
!pip -q install segmentation-models-pytorch onnx onnxscript onnxruntime

In [ ]:
import hashlib, os, pathlib, urllib.request, zipfile

ON_COLAB = pathlib.Path("/content").exists()
BASE = pathlib.Path("/content") if ON_COLAB else pathlib.Path("/kaggle/working")
URL = "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
EXPECTED_MD5 = "c5b1973ade06f7dff210f878161e1a76"

ARCADE_ZIP = BASE / "arcade.zip"
if not ARCADE_ZIP.exists():                     # re-running the cell does not download again
    urllib.request.urlretrieve(URL, ARCADE_ZIP)
digest = hashlib.md5()
with open(ARCADE_ZIP, "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_MD5, f"ARCADE download is corrupt or changed: {digest.hexdigest()}"
DATA = BASE / "arcade_data"
if not DATA.exists():
    zipfile.ZipFile(ARCADE_ZIP).extractall(DATA)
ARC = next(p.parent for p in DATA.rglob("syntax")
           if p.is_dir() and "__MACOSX" not in p.parts and (p / "train").is_dir())
print("ARCADE verified and unpacked at", ARC)

In [ ]:
KIT_DIR = BASE / "kit"
if ON_COLAB:
    from google.colab import files
    uploaded = files.upload()                   # choose mussa-training-kit.zip
    KIT = next(iter(uploaded))
else:                                           # Kaggle: the kit was added as a dataset
    KIT = next(pathlib.Path("/kaggle/input").rglob("mussa-training-kit.zip"))
zipfile.ZipFile(KIT).extractall(KIT_DIR)
print("kit unpacked:", sorted(p.name for p in KIT_DIR.iterdir()))

In [ ]:
# 1) The segment model: where each of the 25 SYNTAX segments is. Rare segments are shown to the model more often
#    (--rare-power) and the loss no longer rewards empty predictions. Add  --smoke  for a one-minute check.
# The best weights are saved as segments_best.pt before the ONNX export, so a failed export loses nothing.
!cd {KIT_DIR} && python train_segmentation.py --task segments --arcade "{ARC}" --out {BASE}/out

In [ ]:
# 2) The stenosis model: where the narrowings are (ARCADE 'stenosis' task, 1000 training images).
!cd {KIT_DIR} && python train_segmentation.py --task stenosis --arcade "{ARC}" --out {BASE}/out

In [ ]:
# OPTIONAL: fine-tune on your own labelled cases. LEAVE THIS OFF unless ALL of these are true:
#   - your hospital's ethics committee has approved using these images for model training and for cloud processing,
#   - the images came from MUSSA's de-identified export (tools/labelling/mussa_labels.py frames), never raw DICOM,
#   - you labelled them and prepared them with `mussa_labels.py split`, giving a zip with train/ and val/ folders.
# Google/Kaggle servers will hold a copy while the notebook runs. When unsure, do not switch it on.
USE_OWN_DATA = False

if USE_OWN_DATA:
    if ON_COLAB:
        from google.colab import files
        mine = next(iter(files.upload()))                # choose labelled.zip
    else:
        mine = next(pathlib.Path("/kaggle/input").rglob("labelled.zip"))
    zipfile.ZipFile(mine).extractall(BASE / "own")
    OWN = BASE / "own"
    # Segments first (start from the ARCADE model, add your own train folder), then the same for stenosis.
    !cd {KIT_DIR} && python train_segmentation.py --task segments --arcade "{ARC}" --out {BASE}/out_ft --init-weights {BASE}/out/segments_best.pt --extra-train {OWN}/train --lr 1e-4 --epochs 30
    !cd {KIT_DIR} && python train_segmentation.py --task stenosis --arcade "{ARC}" --out {BASE}/out_ft --init-weights {BASE}/out/stenosis_best.pt --extra-train {OWN}/train --lr 1e-4 --epochs 30
else:
    print("Skipped: fine-tuning on your own cases is switched off (USE_OWN_DATA = False).")

In [ ]:
import json
import numpy as np
import onnxruntime

FINAL = BASE / ("out_ft" if (BASE / "out_ft" / "segments.onnx").exists() else "out")
print("using models from", FINAL)
for name, channels in (("segments", 25), ("stenosis", 1)):
    card = json.load(open(FINAL / f"{name}.card.json"))
    session = onnxruntime.InferenceSession(str(FINAL / f"{name}.onnx"), providers=["CPUExecutionProvider"])
    size = card["input_size"]
    shape = session.run(None, {session.get_inputs()[0].name: np.zeros((1, 1, size, size), np.float32)})[0].shape
    assert shape == (1, channels, size, size), f"unexpected {name} output shape {shape}"
    print(f"{name}: shape {shape} | validation mean Dice {card['val_mean_dice']:.3f} | test mean Dice {card['test_mean_dice']:.3f}"
          f" | {card['training']['epochs_run']} epochs run")
if ON_COLAB:
    from google.colab import files
    for name in ("segments.onnx", "segments.card.json", "stenosis.onnx", "stenosis.card.json"):
        files.download(str(FINAL / name))
else:
    print("On Kaggle, open the Output tab of this notebook and download the four files from", FINAL)